# 01 --- Synthetic Signal Generation

PINGU generates **7 modulation types** for testing and training the automatic
modulation classifier (AMC). This notebook demonstrates the signal generators
in `pingu.synthetic.signals` and the AWGN noise injection utility in
`pingu.synthetic.noise`.

Supported modulations: **CW, SSB, AM, BPSK, QPSK, FSK2, FSK4**.

Each generator produces a **unit-power complex64** baseband IQ signal,
making it straightforward to combine signals and add calibrated noise at a
desired SNR.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from pingu.synthetic.signals import generate_signal
from pingu.synthetic.noise import add_awgn
from pingu.types import ModulationType

# Reproducible random generator
rng = np.random.default_rng(seed=42)

# Common parameters
SAMPLE_RATE = 48_000  # Hz
DURATION = 0.01       # seconds (480 samples)

print(f"Sample rate : {SAMPLE_RATE} Hz")
print(f"Duration    : {DURATION} s")
print(f"Samples     : {int(SAMPLE_RATE * DURATION)}")

## Signal Generators

The `generate_signal()` dispatcher maps a `ModulationType` enum value to the
correct generator function. All generators accept `sample_rate`, `duration`,
`center_freq`, and an optional `rng` for reproducibility.

In [ ]:
# Generate one example of each modulation type
mod_types = [
    ModulationType.CW,
    ModulationType.SSB,
    ModulationType.AM,
    ModulationType.BPSK,
    ModulationType.QPSK,
    ModulationType.FSK2,
    ModulationType.FSK4,
]

signals = {}
for mod in mod_types:
    sig = generate_signal(
        modulation=mod,
        sample_rate=SAMPLE_RATE,
        duration=DURATION,
        rng=rng,
    )
    signals[mod] = sig
    power = np.mean(np.abs(sig.astype(np.complex128)) ** 2)
    print(f"{mod.name:5s}  len={len(sig):4d}  dtype={sig.dtype}  power={power:.4f}")

In [ ]:
# Plot time-domain waveforms (real part) for all 7 modulation types
fig, axes = plt.subplots(4, 2, figsize=(14, 12))
axes_flat = axes.flatten()

n_samples = int(SAMPLE_RATE * DURATION)
t_ms = np.arange(n_samples) / SAMPLE_RATE * 1000  # time axis in ms

for idx, mod in enumerate(mod_types):
    ax = axes_flat[idx]
    ax.plot(t_ms, np.real(signals[mod]), linewidth=0.8)
    ax.set_title(mod.name, fontsize=12, fontweight="bold")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Re(IQ)")
    ax.grid(True, alpha=0.3)

# Turn off the unused 8th subplot
axes_flat[-1].set_visible(False)

fig.suptitle("Time-Domain Waveforms (Real Part)", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## Spectral Analysis

Power spectral density (PSD) plots reveal the bandwidth and spectral
characteristics of each modulation type. The frequency axis is shown in kHz.

In [ ]:
# Power spectral density for each modulation type
fig, axes = plt.subplots(4, 2, figsize=(14, 12))
axes_flat = axes.flatten()

for idx, mod in enumerate(mod_types):
    ax = axes_flat[idx]
    sig = signals[mod].astype(np.complex128)

    # Compute PSD via FFT
    N = len(sig)
    spectrum = np.fft.fftshift(np.fft.fft(sig))
    psd = 10 * np.log10(np.abs(spectrum) ** 2 / N + 1e-30)
    freqs_khz = np.fft.fftshift(np.fft.fftfreq(N, d=1.0 / SAMPLE_RATE)) / 1000

    ax.plot(freqs_khz, psd, linewidth=0.8)
    ax.set_title(mod.name, fontsize=12, fontweight="bold")
    ax.set_xlabel("Frequency (kHz)")
    ax.set_ylabel("PSD (dB)")
    ax.grid(True, alpha=0.3)

# Turn off the unused 8th subplot
axes_flat[-1].set_visible(False)

fig.suptitle("Power Spectral Density", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## Adding Noise (AWGN)

The `add_awgn()` function injects additive white Gaussian noise at a
specified SNR in dB. The noise power is calibrated so that the output SNR
matches the requested value:

$$P_{\text{noise}} = \frac{P_{\text{signal}}}{10^{\text{SNR}_{\text{dB}}/10}}$$

In [ ]:
# Take the BPSK signal and add AWGN at several SNR levels
bpsk_clean = signals[ModulationType.BPSK]
snr_levels = [20, 10, 0, -5]  # dB

noisy_signals = {}
for snr in snr_levels:
    noisy_signals[snr] = add_awgn(bpsk_clean, snr_db=snr, rng=rng)

# Plot noisy waveforms
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes_flat = axes.flatten()

for idx, snr in enumerate(snr_levels):
    ax = axes_flat[idx]
    ax.plot(t_ms, np.real(noisy_signals[snr]), linewidth=0.8)
    ax.set_title(f"BPSK + AWGN  (SNR = {snr} dB)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Re(IQ)")
    ax.grid(True, alpha=0.3)

fig.suptitle("BPSK Signal with Additive White Gaussian Noise", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# Verify SNR accuracy: compute actual SNR from signal and noise power
print(f"{'Requested SNR (dB)':>20s}  {'Actual SNR (dB)':>16s}  {'Error (dB)':>12s}")
print("-" * 55)

sig_f64 = bpsk_clean.astype(np.complex128)
signal_power = np.mean(np.abs(sig_f64) ** 2)

for snr in snr_levels:
    noisy_f64 = noisy_signals[snr].astype(np.complex128)
    noise_component = noisy_f64 - sig_f64
    noise_power = np.mean(np.abs(noise_component) ** 2)

    actual_snr = 10 * np.log10(signal_power / noise_power) if noise_power > 0 else float("inf")
    error = actual_snr - snr

    print(f"{snr:20d}  {actual_snr:16.2f}  {error:12.2f}")

## Summary

- All seven generators produce **unit-power complex64** baseband IQ signals.
- The `generate_signal()` dispatcher provides a uniform API across modulation types.
- `add_awgn()` injects calibrated noise; actual SNR matches the requested value
  within approximately **1 dB** (limited by finite-length estimation).